# BPE

## 参考

- [Neural Machine Translation of Rare Words with Subword Units][1]
- [Glanguage Models are Unsupervised Multitask Learners][2]
- [openai/gpt-2][3]
- [openai/tiktoken][5]
- [karpathy/minibpe][4]

[1]: https://arxiv.org/abs/1508.07909
[2]: https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf
[3]: https://github.com/openai/gpt-2
[4]: https://github.com/karpathy/minbpe
[5]: https://github.com/openai/tiktoken

## 概要

機械翻訳モデル（NMT, Neural Machine Translation）は、未知語の翻訳が難しかった

この論文では、レアワードをサブワード単位でシーケンスをトークン化し、圧縮するBPE（Byte Pair Encoding）を提案:

- レアワードは、出現頻度の低い単語
- サブワードは、unaffordableを"un", "afford", "able"のような単語の分割

BPEをNMTモデルに使用したところ、BLEUスコア（Bilingual Evaluation Understudy, 翻訳精度）が大幅に向上

## BPE

BPEは、シーケンス中に最も頻繁に出現するバイトのペアを、まだ使われていない単一のバイトに置き換えていくデータ圧縮技術

論文では、バイトのペアではなく、文字または文字シーケンスのペアをマージする

BPEの学習プロセス（Algorithm 1）:

1. 語彙の初期化
    - シンボル語彙（サブワードのリスト）を、データセットに登場するすべての基本文字（アルファベットなど）で初期化
    - 学習データ全体を `low lower` とするとき、 `low lower` -> [`e`, `l`, `o`, `r`, `w`]
2. 単語の表現
    - 学習データ内の単語を文字のシーケンスとして表現
    - 単語の区切りを復元できるように、単語終端記号`</w>`を末尾に追加（シンボル語彙にも追加）
    - `low lower` -> `l o w </w> l o w e r </w>`
3. 反復的なマージ（マージ操作の最大数まで繰り返す）
    1. ペアのカウント
        - 隣り合っているシンボルのペアの出現回数を全てカウント
        - `l o` -> 2, `o w` -> 2, `w </w>` -> 1, ...
    2. 最瀕ペアの特定
        - 最も頻繁に出現するペアを見つける
        - `l o`
    3. マージと置き換え
        - `lo w </w> lo w e r </w>`
    4. シンボル語彙の追加
        - [`e`, `l`, `o`, `r`, `w`, `</w>`, `lo`]

BPEの適用プロセス:

1. 単語を文字のシーケンスに変換
    - `lowest` -> `l o w e s t </w>`
2. 反復的なマージ
    - シンボル語彙を参照して、出現頻度の高かった順に文字をマージ
    - `l, o, w, e, s, t, </w>` -> `lo w e s t </w>`

![](image/algorithm1.png)

BPEが有名になったのは、[GPT-2][1]の論文と実装がきっかけ

[1]: https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf

## 環境構築

In [ ]:
# nanochatのRustBPEをビルドしてインストール

%pip install maturin
import os

if not os.path.exists("nanochat"):
    !git clone https://github.com/karpathy/nanochat

try:
    # Google Colabの場合
    from google.colab import userdata

    # ターミナルでRustをインストール
    # curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y && . "$HOME/.cargo/env"

    # ターミナルでRustBPEのビルドとインストール
    # maturin build --release --manifest-path nanochat/rustbpe/Cargo.toml

    # ターミナルで生成されたwhlファイルをインストール
    # pip install nanochat/rustbpe/target/wheels/*.whl

    if not os.path.exists("nanochat/rustbpe/target"):
        raise FileNotFoundError("Rustのインストールとrustbpeのビルド・インストールをターミナルで実行してください。")

except ImportError:
    # ローカル環境の場合
    !maturin develop --release --manifest-path nanochat/rustbpe/Cargo.toml

import rustbpe

In [ ]:
%pip install -q tiktoken

from multiprocessing import Pool
import os
import pyarrow.parquet as pq
import requests
import tiktoken
import time
import unicodedata
import zipfile

## Tokenizerの試作

### Unicode（ユニコード）

Unicodeは、世界中の文字と数字のマッピングの標準規格

In [ ]:
# ord関数を使用して、文字をコードポイントに変換する
ord("0"), ord("あ"), ord("🙌")

### UTF-8

UTF-8は、Unicode文字をバイト列にエンコードするための手法

In [ ]:
# encode関数を使用して、文字をUTF-8エンコードのバイト列に変換する

"0".encode("utf-8"), "あ".encode("utf-8"), "🙌".encode("utf-8")
# 長さは1から4バイトの可変長

### バイト

1バイトは8ビットで、2^8通りの組み合わせを表現できる

In [ ]:
# list関数を使用して、16進数表記のバイト列を10進数のリストに変換する

list("テスト".encode("utf-8"))

# 最も簡単なトークナイザー
# 圧縮率が低いため実用性が低い

### BPE（Byte-Pair Encoding）

BPE（Byte-Pair Encoding）は、バイトシーケンスを圧縮するアルゴリズム

頻繁に出現するバイトペアをマージすることで高い圧縮率を実現する

In [ ]:
if not os.path.exists("wagahai_ha_nekodearu.txt"):
    !wget -O wagahai_ha_nekodearu.txt https://raw.githubusercontent.com/hayatoshibahara/bpe/refs/heads/main/wagahai_ha_nekodearu.txt

# 青空文庫より抜粋
# https://www.aozora.gr.jp/cards/000148/files/789_14547.html
with open("wagahai_ha_nekodearu.txt", "r", encoding="utf-8") as f:
    long_text = f.read()

# UTF-8でトークン化
tokens = list(long_text.encode("utf-8"))
len(long_text), len(tokens)

# 350439文字 -> 1045557トークン

BPEアルゴリズムを2つの関数を使って実装

In [ ]:
# get_statsは、トークンIDのリストを受け取り、連続するペアの出現回数をカウントして辞書で返す関数

def get_stats(ids, counts=None):
    # 初期値の設定（countsがある場合はそれを使用）
    counts = {} if counts is None else counts

    # Zip関数を使用して、連続する要素のペアを生成
    for pair in zip(ids, ids[1:]): # iterate consecutive elements
        counts[pair] = counts.get(pair, 0) + 1

    return counts

get_stats([1, 2, 3, 1, 2])
# (1, 2)が2回、(2, 3)と(3, 1)が1回

In [ ]:
# mergeは、トークンIDのリストを受け取り、指定されたペアを新しいトークンIDに置換する関数

def merge(ids, pair, idx):
    newids = []
    i = 0
    while i < len(ids):

        # ペアの1文字目と一致し、最後の位置ではなく、ペアの2文字目も一致する場合
        # if not at the very last position AND the pair matches, replace it
        if ids[i] == pair[0] and i < len(ids) - 1 and ids[i+1] == pair[1]:
            # ペアを新しいIDに置き換える
            newids.append(idx)
            i += 2
        else:
            # そのままIDを追加
            newids.append(ids[i])
            i += 1

    return newids

merge([1, 2, 3, 1, 2], (1, 2), 4)
# (1, 2)のペアを新しいIDである4に置き換える

BPEの訓練は、頻繁に出現するバイト値のペアを見つけ、マージし、指定した数まで繰り返すことで行う

In [ ]:
# BPEの訓練

vocab_size = 276  # 語彙サイズ
num_merges = vocab_size - 256 # 最大マージ数は20
tokens = list(long_text.encode("utf-8")) # UTF-8でテキストをバイト列に変換
print(f"Initial token count: {len(tokens)}")

# ペアとマージ後のトークンIDの辞書
merges = {}

# 20回ループ
for i in range(num_merges):

    # すべてのペアをカウント
    stats = get_stats(tokens)

    # 最もカウント数の多いペアを見つける
    pair = max(stats, key=stats.get)

    # 新しいトークンを発行
    idx = 256 + i

    # tokensに含まれるペアを新しいトークンIDで置き換える
    tokens = merge(tokens, pair, idx)

    # 辞書に追加
    merges[pair] = idx

    # 進捗を出力
    print(f"merge {i+1}/{num_merges}: {pair} -> {idx} ({stats[pair]} occurrences)")

print(f"Final token count: {len(tokens)}")
# 1,045,557トークンから660,815トークンに削減

In [ ]:
# ペアとマージ後のトークンIDの辞書
merges

BPEのエンコードは、マージの辞書を使用して行う

In [ ]:
def encode(text):
    # テキストをUTF-8でバイト列に変換
    tokens = list(text.encode("utf-8"))

    # マージできなくなるまで繰り返す
    while len(tokens) >= 2:

        # 連続するペアの出現回数をカウント
        stats = get_stats(tokens)

        # ペアの中から、マージインデックスが最小のものを選ぶ（最も早くマージされたペア）
        pair = min(stats, key=lambda p: merges.get(p, float("inf")))

        # マージインデックスに存在しないペアが終了条件
        if pair not in merges:
            break # これ以上マージできない

        # マージし、tokensを更新
        idx = merges[pair]
        tokens = merge(tokens, pair, idx)
    return tokens

ids = encode("吾輩は猫である")
ids

デコードは、マージの辞書のキーと値を入れ替えたvocab辞書を作成して行う

In [ ]:
# vocab辞書を作成
vocab = {idx: bytes([idx]) for idx in range(256)}
for (p0, p1), idx in merges.items():
    vocab[idx] = vocab[p0] + vocab[p1]

len(vocab)

In [ ]:
def decode(ids):
    # 数字のIDからバイト列に変換
    # given ids (list of integers), return Python string
    tokens = b"".join(vocab[idx] for idx in ids)

    # バイト列をUTF-8でテキストに変換
    # 大規模言語モデルの推論の場合、正しくバイト列を予測できないことがあるのでフォールバック
    text = tokens.decode("utf-8", errors="replace")

    return text

decode(ids)

## 事前トークン化

試作したトークナイザーの場合、「dog.」「dog!」「dog?」は意味が似ているのに独立したトークンとしてマージされてしまう

GPT-2では、文字カテゴリを超えたマージを防ぐ（「dog」と「.」を分ける）

正規表現を使用し、テキストをチャンクのリストに分割する

In [ ]:
import regex as re

# GPT-2の正規表現
pat_gpt2 = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")
# 's|'t|'re|'ve|'m|'ll|'d → 英語の一般的な短縮形
# ?\p{L}+ → オプションのスペース + 1つ以上の文字（Letter）
# ?\p{N}+ → オプションのスペース + 1つ以上の数字（Number）
# ?[^\s\p{L}\p{N}]+ → オプションのスペース + 句読点/記号（空白、文字、数字以外）
# \s+(?!\S)|\s+ → 空白の処理

text = "dog. dog! dog?"
pat_gpt2.findall(text)

## tiktokenライブラリ

OpenAIは、トークナイザーを[OpenAI/tiktoken][1]ライブラリに公開している

開発者はテキストのBPEエンコード・デコードできるが、トークナイザーの訓練はできない

tiktokenの事前トークン化は、tiktoken/tiktoken_ext/openai_public.pyに実装されている

[1]: https://github.com/openai/tiktoken

In [ ]:
from tiktoken.load import data_gym_to_mergeable_bpe_ranks, load_tiktoken_bpe

ENDOFTEXT = "<|endoftext|>"
FIM_PREFIX = "<|fim_prefix|>"
FIM_MIDDLE = "<|fim_middle|>"
FIM_SUFFIX = "<|fim_suffix|>"
ENDOFPROMPT = "<|endofprompt|>"

# GPT-2の事前トークン化の正規表現パターン
# The pattern in the original GPT-2 release is:
# r"""'s|'t|'re|'ve|'m|'ll|'d| ?[\p{L}]+| ?[\p{N}]+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
# This is equivalent, but executes faster:
r50k_pat_str = (
    r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}++| ?\p{N}++| ?[^\s\p{L}\p{N}]++|\s++$|\s+(?!\S)|\s"""
)

# GPT-2のトークナイザー設定
def gpt2():

    # 語彙ファイルはAzureのパブリックストレージからダウンロード
    mergeable_ranks = data_gym_to_mergeable_bpe_ranks(
        vocab_bpe_file="https://openaipublic.blob.core.windows.net/gpt-2/encodings/main/vocab.bpe",
        encoder_json_file="https://openaipublic.blob.core.windows.net/gpt-2/encodings/main/encoder.json",
        vocab_bpe_hash="1ce1664773c50f3e0cc8842619a93edc4624525b728b188a9e0be33b7726adc5",
        encoder_json_hash="196139668be63f3b5d6574427317ae82f612a97c5d1cdaf36ed2256dbf636783",
    )

    return {
        "name": "gpt2",
        "explicit_n_vocab": 50257,
        "pat_str": r50k_pat_str, # 50kはおおよその語彙数
        "mergeable_ranks": mergeable_ranks,
        "special_tokens": {ENDOFTEXT: 50256}, # 文の終わりを示す特殊トークンがある
    }


# GPT-4のトークナイザー設定
def cl100k_base():

    # 語彙ファイル
    mergeable_ranks = load_tiktoken_bpe(
        "https://openaipublic.blob.core.windows.net/encodings/cl100k_base.tiktoken",
        expected_hash="223921b76ee99bde995b7ff738513eef100fb51d18c93597a113bcffe865b2a7",
    )

    # GPT-4は複数の特殊トークンを仕様
    special_tokens = {
        ENDOFTEXT: 100257,
        FIM_PREFIX: 100258,
        FIM_MIDDLE: 100259,
        FIM_SUFFIX: 100260,
        ENDOFPROMPT: 100276,
    }

    return {
        "name": "cl100k_base", # 100kはおおよその語彙数
        # GPT-4の事前トークン化の正規表現パターン
        "pat_str": r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}++|\p{N}{1,3}+| ?[^\s\p{L}\p{N}]++[\r\n]*+|\s++$|\s*[\r\n]|\s+(?!\S)|\s""",
        "mergeable_ranks": mergeable_ranks,
        "special_tokens": special_tokens,
    }


## GPT-2とGPT-4の事前トークン化

In [ ]:
# GPT-2とGPT-4のトークン化を比較

enc_gpt2 = tiktoken.get_encoding("gpt2")
enc_gpt4 = tiktoken.get_encoding("cl100k_base")

text = "Hello,             world!"
tokens_gpt2 = enc_gpt2.encode(text)
tokens_gpt4 = enc_gpt4.encode(text)

print(f"GPT-2: {tokens_gpt2}")
print(f"GPT-4: {tokens_gpt4}")

# GPT-2は、空白を1つのトークンとして扱う（220）
# GPT-4は、空白をマージする（1078）


In [ ]:
# GPT-2とGPT-4の事前正規化のスペースの扱いを比較

gpt2_pat = re.compile(r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}++| ?\p{N}++| ?[^\s\p{L}\p{N}]++|\s++$|\s+(?!\S)|\s""")
gpt4_pat = re.compile(r"""(?i:'s|'t|'re|'ve|'m|'ll|'d)|[^\r\n\p{L}\p{N}]?\p{L}+|\p{N}{2,}|[^\r\n\p{L}\p{N}]?[^\s\p{L}\p{N}]+[\r\n]*|\s*[\r\n]+|\s+(?!\S)|\s+""")

text = "Hello,            world!"
matches_gpt2 = gpt2_pat.findall(text)
matches_gpt4 = gpt4_pat.findall(text)

print(f"GPT-2: {matches_gpt2}")
print(f"GPT-4: {matches_gpt4}")

In [ ]:
# GPT-2とGPT-4の事前正規化の英語の短縮形の扱いを比較

text = "HOW'S IT GOING? how's it going?"
gpt2_result = gpt2_pat.findall(text)
gpt4_result = gpt4_pat.findall(text)

print(f"GPT-2: {gpt2_result}")
print(f"GPT-4: {gpt4_result}")

# 大文字の場合は、GPT-2はアポストロフィーの扱いが異なる

In [ ]:
# GPT-2とGPT-4の事前正規化の数字の扱いを比較

test_numbers = "I have 2 apples, 12 oranges, and 12345678 bananas."
gpt2_result = gpt2_pat.findall(test_numbers)
gpt4_result = gpt4_pat.findall(test_numbers)

print(f"GPT-2: {gpt2_result}")
print(f"GPT-4: {gpt4_result}")

# 共通して2桁以上の数字を一つのトークンにまとめる
# GPT-4は、1桁の数字を無視

In [ ]:
# GPT-2とGPT-4の事前正規化の改行と空白の扱いを比較

test_newlines = "Hello\nworld\n\n  \ntest"
gpt2_result = gpt2_pat.findall(test_newlines)
gpt4_result = gpt4_pat.findall(test_newlines)
print(f"GPT-2: {gpt2_result}")
print(f"GPT-4: {gpt4_result}")

# GPT-4は、複数の改行と空白を一つのトークンにまとめる

## Tokenizer

Tokenizerは、トークナイザーのベースクラス

In [ ]:
# replace_control_charactersは、文字列中の制御文字をエスケープする関数
# 制御文字は改行（\n）やタブ（\t）などで、人が確認するためにエスケープ
# 訓練済みのBPEで語彙ファイル保存時に使用

def replace_control_characters(s: str) -> str:
    chars = []
    for ch in s:
        # 制御文字（Control Character）でない場合
        if unicodedata.category(ch)[0] != "C":
            # そのまま追加
            chars.append(ch) # this character is ok
        else:
            # ユニコードエスケープ形式に置き換えて追加
            chars.append(f"\\u{ord(ch):04x}") # escape

    return "".join(chars)

replace_control_characters("Hello\nWorld\t!")

In [ ]:
# render_tokenは、バイト列をテキストに変換し、制御文字をエスケープする関数
# 語彙ファイル保存時に使用

def render_token(t: bytes) -> str:
    # バイト列をテキストに変換
    # マージされたバイト列は変換できないこともあるため、フォールバックを使用
    s = t.decode('utf-8', errors='replace')

    # 制御文字をエスケープ
    s = replace_control_characters(s)
    return s

render_token(b'Hello\nWorld\t!')

In [ ]:
class Tokenizer:

    def __init__(self):
        # default: vocab size of 256 (all bytes), no merges, no patterns

        # マージルールの辞書
        # 例: {(65, 66): 256}は、A(65)とB(66)が隣り合っていたらAB(256)にする
        self.merges = {} # (int, int) -> int

        # RegexTokenizerで使用する正規表現パターン
        self.pattern = ""

        # 特殊トークンの辞書
        # 例: {'<|endoftext|>': 100257}
        self.special_tokens = {}

        # 語彙の辞書（ID -> バイト列）
        self.vocab = self._build_vocab()

    def train(self, text, vocab_size, verbose=False):
        raise NotImplementedError

    def encode(self, text):
        raise NotImplementedError

    def decode(self, ids):
        raise NotImplementedError

    def _build_vocab(self):
        """
        self.mergeとself.special_tokensからself.vocabを構築する内部メソッド
        vocabは、IDからバイト列への辞書
        """
        # 初期の256バイトを語彙に追加
        vocab = {idx: bytes([idx]) for idx in range(256)}

        # マージルールを使用して、マージしたバイト列を辞書に追加
        for (p0, p1), idx in self.merges.items():
            vocab[idx] = vocab[p0] + vocab[p1]

        # 特殊トークンを辞書に追加
        for special, idx in self.special_tokens.items():
            vocab[idx] = special.encode("utf-8")

        return vocab

    def save(self, file_prefix):
        """
        2種類のファイルを保存:
        1. .model: トークナイザーを復元するために使用で、正規表現・特殊トークン・マージルールを含む
        2. .vocab: 人間が確認するための可読形式
        """

        # 1. modelファイルを作成

        model_file = file_prefix + ".model"

        with open(model_file, 'w') as f:
            # トークナイザーのバージョン
            f.write("minbpe v1\n")
            # 正規表現
            f.write(f"{self.pattern}\n")
            # 特殊トークンの数
            f.write(f"{len(self.special_tokens)}\n")
            # すべての特殊トークン
            for special, idx in self.special_tokens.items():
                f.write(f"{special} {idx}\n")
            # すべてのマージルール
            for idx1, idx2 in self.merges:
                f.write(f"{idx1} {idx2}\n")

        # 2. vocabファイルを作成

        vocab_file = file_prefix + ".vocab"

        # マージルールの逆引き辞書を作成（マージ後のID -> マージ前のペア）
        inverted_merges = {idx: pair for pair, idx in self.merges.items()}

        with open(vocab_file, "w", encoding="utf-8") as f:
            for idx, token in self.vocab.items():
                # バイト列をテキストに変換
                s = render_token(token)

                # マージ前のトークンがある場合
                if idx in inverted_merges:
                    idx0, idx1 = inverted_merges[idx]
                    s0 = render_token(self.vocab[idx0])
                    s1 = render_token(self.vocab[idx1])
                    # マージ前のトークンも表示
                    f.write(f"[{s0}][{s1}] -> [{s}] {idx}\n")
                else:
                    # そのまま表示
                    f.write(f"[{s}] {idx}\n")

    def load(self, model_file):
        """
        .modelファイルからトークナイザーを復元するメソッド
        """
        assert model_file.endswith(".model")

        merges = {}
        special_tokens = {}
        idx = 256

        with open(model_file, 'r', encoding="utf-8") as f:
            # バージョン
            version = f.readline().strip()
            assert version == "minbpe v1"

            # 正規表現
            self.pattern = f.readline().strip()

            # 特殊トークンの数
            num_special = int(f.readline().strip())

            # 特殊トークンを読み込み
            for _ in range(num_special):
                special, special_idx = f.readline().strip().split()
                special_tokens[special] = int(special_idx)

            # マージルールを読み込み
            for line in f:
                idx1, idx2 = map(int, line.split())
                merges[(idx1, idx2)] = idx
                idx += 1

        # プロパティを更新
        self.merges = merges
        self.special_tokens = special_tokens
        self.vocab = self._build_vocab()

## BasicTokenizer

BasicTokenizerは、GPT-2を参考にした最もシンプルなBPEトークナイザークラス

事前トークン化と特殊トークンは扱っていない

In [ ]:
class BasicTokenizer(Tokenizer):

    def __init__(self):
        super().__init__()

    def train(self, text, vocab_size, verbose=False):
        assert vocab_size >= 256
        num_merges = vocab_size - 256

        # 文字列をバイト列に変換
        text_bytes = text.encode("utf-8")

        # 10進数のリストに変換（0から255の整数）
        ids = list(text_bytes)

        # マージの辞書を初期化
        merges = {} # (int, int) -> int

        # 語彙を初期化（int -> bytes）
        vocab = {idx: bytes([idx]) for idx in range(256)}

        # num_mergesまでマージを繰り返す
        for i in range(num_merges):

            # すべてのペアの出現頻度をカウント
            stats = get_stats(ids)

            # 最もカウント数の多いペアを見つける
            pair = max(stats, key=stats.get)

            # 新しいトークンIDを発行
            idx = 256 + i

            # tokensに含まれるペアを新しいIDで置き換える
            ids = merge(ids, pair, idx)

            # 辞書に追加
            merges[pair] = idx

            # 語彙に追加
            vocab[idx] = vocab[pair[0]] + vocab[pair[1]]

            # 進捗を出力
            if verbose:
                print(f"merge {i+1}/{num_merges}: {pair} -> {idx} ({vocab[idx]}) had {stats[pair]} occurrences")

        # マージルールと語彙をプロパティに保存
        self.merges = merges
        self.vocab = vocab

    def encode(self, text):
        # テキストをUTF-8でバイト列に変換
        text_bytes = text.encode("utf-8")

        # 10進数のリストに変換
        ids = list(text_bytes)

        # マージできなくなるまで繰り返す
        while len(ids) >= 2:

            # 連続するペアの出現回数をカウント
            stats = get_stats(ids)

            # ペアの中から、マージインデックスが最小のものを選ぶ（最も早くマージされたペア）
            pair = min(stats, key=lambda p: self.merges.get(p, float("inf")))

            # マージインデックスに存在しないペアが終了条件
            if pair not in self.merges:
                break # これ以上マージできない

            # マージし、idsを更新
            idx = self.merges[pair]
            ids = merge(ids, pair, idx)
        return ids

    def decode(self, ids):
        # IDをバイト列に変換
        text_bytes = b"".join(self.vocab[idx] for idx in ids)

        # バイト列をUTF-8でテキストに変換
        text = text_bytes.decode("utf-8", errors="replace")
        return text

# 訓練
basic_tokenizer = BasicTokenizer()
basic_tokenizer.train(long_text, vocab_size=276, verbose=True)

In [ ]:
token = basic_tokenizer.encode("吾輩は猫である。")
len(token), token

In [ ]:
basic_tokenizer.decode(token)

In [ ]:
basic_tokenizer.save("basic_tokenizer")

# basic_tokenizer.modelは、読み込み用のモデルファイル
# basic_tokenizer.vocabは、人間が読んで確認するための語彙ファイル

In [ ]:
basic_tokenizer.load("basic_tokenizer.model")

# 読み込み

## RegexTokenizer

RegexTokenizerは、事前トークン化と特殊トークン（special tokens）を扱う一般的なトークナイザー

事前トークン化により、文字カテゴリの境界を超えてマージされないように設計されている

GPT-2・GPT-4でも導入されている

特殊トークンは、文の区切りや会話の構造タグを示すトークンで、ファインチューニング時に主に追加される

In [ ]:
import regex as re

# 事前トークン化のための正規表現
GPT2_SPLIT_PATTERN = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
GPT4_SPLIT_PATTERN = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""

# Tokenizerクラスを継承
class RegexTokenizer(Tokenizer):

    def __init__(self, pattern=None):
        super().__init__()

        # 事前トークン化の正規表現パターンを設定
        self.pattern = GPT4_SPLIT_PATTERN if pattern is None else pattern

        # 正規表現をコンパイル
        self.compiled_pattern = re.compile(self.pattern)

        # 特殊トークンの辞書（str -> int）
        self.special_tokens = {}

        # 特殊トークンの逆引き辞書（int -> str）
        self.inverse_special_tokens = {}

    def train(self, text, vocab_size, verbose=False):
        assert vocab_size >= 256
        num_merges = vocab_size - 256

        # 正規表現でテキストをチャンクに分割
        text_chunks = re.findall(self.compiled_pattern, text)

        # チャンクをバイト列に変換し、10進数のリストに変換
        ids = [list(ch.encode("utf-8")) for ch in text_chunks]

        # マージルールの初期化
        merges = {} # (int, int) -> int

        # 語彙の初期化
        vocab = {idx: bytes([idx]) for idx in range(256)} # idx -> bytes

        for i in range(num_merges):

            # 連続するペアの出現回数をチャンクをまたいでカウント
            stats = {}
            for chunk_ids in ids:
                # チャンクごとにペアの出現数をカウントし、statsを更新
                get_stats(chunk_ids, stats)

            # 最もカウント数の多いペアを見つける
            pair = max(stats, key=stats.get)

            # 新しいトークンIDを発行
            idx = 256 + i

            # チャンクごとにペアを新しいIDで置き換える
            ids = [merge(chunk_ids, pair, idx) for chunk_ids in ids]

            # マージルールを更新
            merges[pair] = idx

            # 語彙を更新
            vocab[idx] = vocab[pair[0]] + vocab[pair[1]]

            # 進捗を出力
            if verbose:
                print(f"merge {i+1}/{num_merges}: {pair} -> {idx} ({vocab[idx]}) had {stats[pair]} occurrences")

        # プロパティにマージルールと語彙を保存
        self.merges = merges
        self.vocab = vocab

    def register_special_tokens(self, special_tokens):
        """
        特殊トークンを登録するメソッド
        特殊トークンはエンコード・デコード時に使用する
        例: {"<|endoftext|>": 100257}
        """
        # プロパティに保存
        self.special_tokens = special_tokens

        # 逆引き辞書も作成して保存
        self.inverse_special_tokens = {v: k for k, v in special_tokens.items()}

    def _encode_chunk(self, text_bytes):
        """
        テキストチャンクのバイト列を受け取り、トークンIDのリストを返す内部メソッド
        """

        # バイト列を10進数のリストに変換
        ids = list(text_bytes)

        # マージできなくなるまで繰り返す
        while len(ids) >= 2:

            # 連続するペアの出現回数をカウント
            stats = get_stats(ids)

            # ペアの中から、マージインデックスが最小のものを選ぶ（最も早くマージされたペア）
            pair = min(stats, key=lambda p: self.merges.get(p, float("inf")))

            # マージできなくなったら終了
            if pair not in self.merges:
                break

            # マージし、トークンIDのリストを更新
            idx = self.merges[pair]
            ids = merge(ids, pair, idx)
        return ids

    def encode_ordinary(self, text):
        """
        特殊トークンを考慮しない場合のエンコードメソッド
        """

        # 事前トークン化し、テキストをチャンクに分割
        text_chunks = re.findall(self.compiled_pattern, text)

        # チャンクごとにエンコードし、結果を結合
        ids = []
        for chunk in text_chunks:
            # テキストチャンクをバイト列に変換
            chunk_bytes = chunk.encode("utf-8")

            # バイト列をトークンIDに変換
            chunk_ids = self._encode_chunk(chunk_bytes)

            # 結果を結合
            ids.extend(chunk_ids)
        return ids

    def encode(self, text, allowed_special="none_raise", verbose=False):
        """
        特殊トークンを考慮する場合のエンコードメソッド

        allowed_special: 特殊トークンの扱いを指定
        - "all": すべての特殊トークンを許可
        - "none": 特殊トークンを無視
        - "none_raise": 特殊トークンが含まれている場合はエラーを発生させる
        - set型: 指定された特殊トークンのみを許可
        """

        # decode the user desire w.r.t. handling of special tokens

        # 1. 特殊トークンの扱いを決定
        special = None

        # allの場合
        if allowed_special == "all":
            # すべての特殊トークンを許可
            special = self.special_tokens

        # noneの場合
        elif allowed_special == "none":
            # 特殊トークンを無視
            special = {}

        # none_raiseの場合
        elif allowed_special == "none_raise":
            special = {}
            # 特殊トークンが含まれている場合はエラーを発生させる
            assert all(token not in text for token in self.special_tokens)

        # 特殊トークンがセットで与えられた場合
        elif isinstance(allowed_special, set):
            # 指定された特殊トークンのみを許可
            special = {k: v for k, v in self.special_tokens.items() if k in allowed_special}

        # その他は例外
        else:
            raise ValueError(f"allowed_special={allowed_special} not understood")

        # 特殊トークンがない場合は、普通にエンコード
        if not special:
            return self.encode_ordinary(text)

        # 特殊トークン用の正規表現パターンを作成
        special_pattern = "(" + "|".join(re.escape(k) for k in special) + ")"

        if verbose:
            print(f"special token pattern: {special_pattern}")

        # 特殊トークン用の正規表現でテキストをチャンクに分割
        special_chunks = re.split(special_pattern, text)

        ids = []
        for part in special_chunks:
            # チャンクが特殊トークンの場合
            if part in special:
                # 特殊トークンのIDを追加
                ids.append(special[part])
            else:
                # チャンクを普通にエンコード
                ids.extend(self.encode_ordinary(part))
        return ids

    def decode(self, ids):

        # バッファを初期化
        part_bytes = []

        for idx in ids:
            # IDが語彙に含まれる場合
            if idx in self.vocab:
                # 語彙からバイト列を取得し、バッファに追加
                part_bytes.append(self.vocab[idx])

            # IDが特殊トークンに含まれる場合
            elif idx in self.inverse_special_tokens:
                # 特殊トークンの逆引き辞書からバイト列を取得し、バッファに追加
                part_bytes.append(self.inverse_special_tokens[idx].encode("utf-8"))
            else:
                raise ValueError(f"invalid token id: {idx}")

        # バイト列を結合
        text_bytes = b"".join(part_bytes)

        # バイト列を文字列に変換
        text = text_bytes.decode("utf-8", errors="replace")
        return text

regex_tokenizer = RegexTokenizer()
regex_tokenizer.train(long_text, vocab_size=276, verbose=True)

In [ ]:
token = regex_tokenizer.encode("吾輩は猫である。")
len(token), token

In [ ]:
regex_tokenizer.decode(token)

In [ ]:
regex_tokenizer.save("regex_tokenizer")

In [ ]:
regex_tokenizer.load("regex_tokenizer.model")

## GPT4Tokenizer

GPT4Tokenizerは、BPEの前にバイトシャッフルを適用するGPT-4のトークナイザー

バイトシャッフルは、バイト値を特定の順番

訓練済みのマージルールをOpenAIのサーバーからダウンロードし、Tokenizerクラス用に変換し、再現してみる

In [ ]:
# 訓練済みのマージルールのダウンロード

# GPT-4のマージルールをダウンロード
enc = tiktoken.get_encoding("cl100k_base")
enc.name, enc.max_token_value

In [ ]:
# 事前トークン化の正規表現パターン
enc._pat_str

In [ ]:
# マージルールの辞書（バイト列 -> トークンID）
# tiktokenではトークンIDは、マージの優先順位ランクを示す
# 値が小さいほど優先的にマージされる
enc._mergeable_ranks

import random
rand_index = random.randint(0, enc.max_token_value - 10)
list(enc._mergeable_ranks.items())[rand_index:(rand_index + 10)]

In [ ]:
# 特殊トークンの辞書（文字列 -> トークンID）
enc._special_tokens

In [ ]:
# bpeは、1つのトークンをマージルールに従ってmax_rankまで分解する関数
# tiktoken用マージルールをTokenizerクラス用マージルールに変換するrecover_merge関数で使用する

def bpe(mergeable_ranks, token, max_rank):

    # トークンIDをバイト列に変換し、1バイトずつのリストに分解
    parts = [bytes([b]) for b in token]

    # マージルールに従ってマージ
    while True:
        min_idx = None
        min_rank = None

        # 連続するペアの中で、最もランクが小さいペアを見つける
        for i, pair in enumerate(zip(parts[:-1], parts[1:])):
            rank = mergeable_ranks.get(pair[0] + pair[1])
            if rank is not None and (min_rank is None or rank < min_rank):
                min_idx = i
                min_rank = rank

        # マージできなくなる、または最大ランクに達したら終了
        if min_rank is None or (max_rank is not None and min_rank >= max_rank):
            break

        assert min_idx is not None

        # 最もランクが小さいペアをマージ
        parts = parts[:min_idx] + [parts[min_idx] + parts[min_idx + 1]] + parts[min_idx + 2:]

    return parts

print(bpe(enc._mergeable_ranks, b"HuggingFace", 100))
print(bpe(enc._mergeable_ranks, b"HuggingFace", 1000))
print(bpe(enc._mergeable_ranks, b"HuggingFace", 10000))
print(bpe(enc._mergeable_ranks, b"HuggingFace", None))

In [ ]:
# recover_mergeは、tiktoken用マージルールをTokenizerクラス用マージルールに変換する関数
# tiktoken用マージルール: {(b'abo', 48521), ...}
# Tokenizer用マージルール: {(97, 98): 256, ...}

def recover_merges(mergeable_ranks):
    merges = {}

    # すべてのマージ可能なトークンを処理
    for token, rank in mergeable_ranks.items():

        # 長さが1の場合はスキップ
        if len(token) == 1:
            continue

        # tokenをmax_rank=rankでbpe分解し、ペアを取得
        pair = tuple(bpe(mergeable_ranks, token, max_rank=rank))

        assert len(pair) == 2

        # ペアを復元
        ix0 = mergeable_ranks[pair[0]]
        ix1 = mergeable_ranks[pair[1]]
        merges[(ix0, ix1)] = rank

    return merges

In [ ]:
GPT4_SPLIT_PATTERN = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""

GPT4_SPECIAL_TOKENS = {
    '<|endoftext|>': 100257,
    '<|fim_prefix|>': 100258,
    '<|fim_middle|>': 100259,
    '<|fim_suffix|>': 100260,
    '<|endofprompt|>': 100276
}

# RegexTokenierを継承
class GPT4Tokenizer(RegexTokenizer):

    def __init__(self):
        super().__init__(pattern=GPT4_SPLIT_PATTERN)

        # tiktokenのマージルールをダウンロード
        enc = tiktoken.get_encoding("cl100k_base")
        mergeable_ranks = enc._mergeable_ranks

        # Tokenizerクラス用のマージルールに復元
        self.merges = recover_merges(mergeable_ranks)

        # 語彙を初期化（ID -> バイト列）
        vocab = {idx: bytes([idx]) for idx in range(256)}

        # マージルールから語彙を復元
        for (p0, p1), idx in self.merges.items():
            vocab[idx] = vocab[p0] + vocab[p1]

        # 語彙をプロパティに設定
        self.vocab = vocab

        # 0~255のバイト値をGPT-4のバイトシャッフルに従ってマッピング
        self.byte_shuffle = {i: mergeable_ranks[bytes([i])] for i in range(256)}

        # バイトシャッフルの逆引き辞書も作成
        self.inverse_byte_shuffle = {v: k for k, v in self.byte_shuffle.items()}

        # GPT-4の特殊トークンを登録
        self.register_special_tokens(GPT4_SPECIAL_TOKENS)

    def _encode_chunk(self, text_bytes):
        """
        テキストチャンクのバイト列を受け取り、トークンIDのリストを返す内部メソッド
        """
        # バイトシャッフルでtiktokenの内部的な順番に並び替え
        text_bytes = bytes(self.byte_shuffle[b] for b in text_bytes)
        ids = super()._encode_chunk(text_bytes)
        return ids

    def decode(self, ids):
        # IDをバイト列に変換
        text_bytes = b"".join(self.vocab[idx] for idx in ids)

        # バイトシャッフルの逆引きで元のバイト順に戻す
        text_bytes = bytes(self.inverse_byte_shuffle[b] for b in text_bytes)

        # バイト列を文字列に変換
        text = text_bytes.decode("utf-8", errors="replace")
        return text

    def train(self, text, vocab_size, verbose=False):
        raise NotImplementedError

    def save(self, file_prefix):
        raise NotImplementedError("GPT4Tokenizer cannot be saved.")

    def load(self, model_file):
        raise NotImplementedError("GPT4Tokenizer cannot be loaded.")

    def save_vocab(self, vocab_file):
        """
        人が確認するための語彙ファイルを保存するメソッド
        形式はBaseTokenizerと同じ
        """
        # 初期の256バイトをシャッフルして語彙に追加
        vocab = {idx: bytes([self.inverse_byte_shuffle[idx]]) for idx in range(256)}

        # マージルールを使用して、マージしたバイト列を辞書に追加
        for (p0, p1), idx in self.merges.items():
            vocab[idx] = vocab[p0] + vocab[p1]

        # マージルールの逆引き辞書を作成（マージ後のID -> マージ前のペア）
        inverted_merges = {idx: pair for pair, idx in self.merges.items()}

        # 語彙ファイルを作成
        with open(vocab_file, "w", encoding="utf-8") as f:
            # すべてのトークンをID順に出力
            for idx, token in vocab.items():
                # バイト列を人が読める形式に変換
                s = render_token(token)

                # マージ前のトークンがある場合
                if idx in inverted_merges:
                    # 分解して出力
                    idx0, idx1 = inverted_merges[idx]
                    s0 = render_token(vocab[idx0])
                    s1 = render_token(vocab[idx1])
                    f.write(f"[{s0}][{s1}] -> [{s}] {idx}\n")
                else:
                    # そのまま出力
                    f.write(f"[{s}] {idx}\n")

gpt4_tokenizer = GPT4Tokenizer()
gpt4_tokenizer.save_vocab("gpt4.vocab")

In [ ]:
# GPT-4トークナイザーと同じ結果が得られる

text = "hello123!!!? (안녕하세요!) 😉"
enc = tiktoken.get_encoding("cl100k_base")
tokenizer = GPT4Tokenizer()

print("tiktoken", enc.encode(text))
print("GPT4Tokenizer", tokenizer.encode(text))

In [ ]:
# 特殊トークンも同様

text = "<|endoftext|>hello world"
enc = tiktoken.get_encoding("cl100k_base")
tokenizer = GPT4Tokenizer()

print("tiktoken", enc.encode(text, allowed_special="all"))
print("GPT4Tokenizer", tokenizer.encode(text, allowed_special="all"))

## RustbpeTokenizer

`nanochat/rustbpe/src/lib.rs`より抜粋し、その冒頭から解説

### 1. ライブラリのインポート

```rust
// Ordering型・HashMap型
use std::cmp::Ordering;
use std::collections::HashMap as StdHashMap;

// 8分ヒープ
// 最小値・最大値を効率的に取り出す木構造（8分ヒープは、最大8個の子ノード）
use dary_heap::OctonaryHeap;

// 正規表現
use fancy_regex::Regex;

// PythonとRustを連携させるための基本機能（prelude）
use pyo3::prelude::*;

// 高速なハッシュマップ
use ahash::{AHashMap, AHashSet};

// 短い文字列を効率よく保存できる文字列型
use compact_str::CompactString;

// 並列処理の基本機能
use rayon::prelude::*;
```

### 2. リテラル・型・構造体の定義

```rust
// GPT4の事前トークン化の正規表現
const GPT4_PATTERN: &str = r"'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+";

// BPEのトークンIDのペアのエイリアス型
type Pair = (u32, u32);

// Tokenizer構造体の定義
#[pyclass] // import rustbpe のように呼び出すためのアトリビュート
pub struct Tokenizer {

    pub merges: StdHashMap<Pair, u32>, // ペアと新しいトークンIDのマップ（マージルール）

    pub pattern: String, // 正規表現パターン

    compiled_pattern: Regex, // コンパイル後の正規化パターン
}
```

```rust

// トークンIDのリストを保持するWord構造体を定義
#[derive(Clone, Debug)]
struct Word {
    ids: Vec<u32>,
}

// Word構造体を実装
impl Word {

    // コンストラクタ
    #[inline]
    fn new(ids: Vec<u32>) -> Self {
        Self { ids }
    }

    // 隣り合う全てのペアのリストを返す
    #[inline]
    fn pairs<'a>(&'a self) -> impl Iterator<Item = Pair> + 'a {
        self.ids.windows(2).map(|w| (w[0], w[1]))
    }

    // ペアをマージする
    fn merge_pair(&mut self, pair: Pair, new_id: u32) -> Vec<(Pair, i32)> {

        let (a, b) = pair;

        // このワードのIDの長さ
        let n = self.ids.len();

        // ペアが存在しない場合は空の配列を返す
        if n < 2 {
            return Vec::new();
        }

        // マージ後のIDを入れるための配列を初期化
        let mut out: Vec<u32> = Vec::with_capacity(n);

        // ペアの増減操作を記録するための配列を初期化
        let mut deltas: Vec<(Pair, i32)> = Vec::with_capacity(6);

        // ワードのIDを先頭からループ
        let mut i = 0;
        while i < n {

            // aとbが連続した並びのインデックスの場合（ペアが見つかった場合）
            if i + 1 < n && self.ids[i] == a && self.ids[i + 1] == b {

                // self.ids = [5, 10, 20, 30], pair = (10, 20) のとき left = Some(5)
                let left = out.last().copied();

                // self.ids = [5, 10, 20, 30], pair = (10, 20) のとき right = Some(30)
                let right = if i + 2 < n { Some(self.ids[i + 2]) } else { None };

                // new_id = 500 のとき
                if let Some(x) = left {
                    deltas.push(((x, a), -1)); // ((5, 10), -1) をdeltaに追加
                    deltas.push(((x, new_id), 1)); // (5, 500) をdeltaに追加
                }

                deltas.push(((a, b), -1)); // ((10, 20), -1) をdeltaに追加

                if let Some(y) = right {
                    deltas.push(((b, y), -1)); // ((20, 30), -1) をdeltaに追加
                    deltas.push(((new_id, y), 1)); // ((500, 20), -1) をdeltaに追加
                }

                // out に 500 を追加
                out.push(new_id);

                // カウンターを2つずらす
                i += 2;
            } else {
                // マージ後のIDのリストで、ワードのIDのリストを置き換える
                out.push(self.ids[i]);

                // カウンターを1ずらす
                i += 1;
            }
        }

        // マージ後のIDのリストで、ワードのIDのリストを置き換える
        self.ids = out;

        // deltasを返す
        deltas
    }
}
```

MergeJobは、BPEの訓練時にマージを効率的に行うジョブチケット

```rust
#[derive(Debug, Eq)]
struct MergeJob {
    pair: Pair, // トークンのペア
    count: u64, // データセット全体で出現する回数
    pos: AHashSet<usize>, // トークンのペアが出現する可能性のあるWordのインデックス
}

// 等価性のルール（==）
impl PartialEq for MergeJob {
    fn eq(&self, other: &Self) -> bool {
        // 出現回数とペアが同じ場合、等価
        self.count == other.count && self.pair == other.pair
    }
}

// 序列のルール（<, >）
impl Ord for MergeJob {
    fn cmp(&self, other: &Self) -> Ordering {
        // カウント数が異なる場合
        if self.count != other.count {
            // カウントが小さい方を優先
            self.count.cmp(&other.count)
        } else {
            // カウントが同じ場合は、IDが小さい方を優先
            other.pair.cmp(&self.pair)
        }
    }
}

// 部分的な序列のルール（<=, =>）
impl PartialOrd for MergeJob {
    fn partial_cmp(&self, other: &Self) -> Option<Ordering> {
        // Ordのself.cmpを使用
        Some(self.cmp(other))
    }
}


```

### 3. ヘルパー関数

count_pairs_parallelは、rayonライブラリで全CPUコアを使用し、高速にペアを集計する関数

テキスト全体のペアを数えるのではなく、$\text{Wordの出現回数} \times \text{Word内のペア数}$ で計算

```rust
#[inline]
fn count_pairs_parallel(
    words: &[Word], // Wordの巨大なリスト
    counts: &[i32], // Wordの出現回数のリスト（ペアではない）
) -> (AHashMap<Pair, i32>, AHashMap<Pair, AHashSet<usize>>) {
    words
        .par_iter() // rayonの並列イテレータ
        .enumerate()
        .map(|(i, w)| {

            // ペアの出現回数（pair count）
            let mut local_pc: AHashMap<Pair, i32> = AHashMap::new();

            // ペアが含まれるワードのインデックス（word to update）
            let mut local_wtu: AHashMap<Pair, AHashSet<usize>> = AHashMap::new();

            if w.ids.len() >= 2 && counts[i] != 0 {

                // ワード内のペアでループ
                for (a, b) in w.pairs() {
                    *local_pc.entry((a, b)).or_default() += counts[i]; // Wordの出現回数を加算
                    local_wtu.entry((a, b)).or_default().insert(i); // インデックスを追加
                }
            }

            (local_pc, local_wtu)
        })
        .reduce( // 全スレッドの結果を集約
            || (AHashMap::new(), AHashMap::new()),
            |(mut acc_pc, mut acc_wtu), (pc, wtu)| {
                for (k, v) in pc {
                    *acc_pc.entry(k).or_default() += v; // アキュームレータに出現回数を累積
                }
                for (k, s) in wtu {
                    acc_wtu.entry(k).or_default().extend(s); // アキュームレータにインデックスを累積
                }
                (acc_pc, acc_wtu)
            },
        )
}
```

### 4. Tokenizer（内部）

Tokenizerは、BPEの訓練を実行する関数

```rust
impl Tokenizer {

    // Wordのリストとその出現回数を受け取り、出現頻度の高いペアをマージし、マージルールを作成する 
    fn train_core_incremental(&mut self, mut words: Vec<Word>, counts: Vec<i32>, vocab_size: u32) {

        assert!(vocab_size >= 256, "vocab_size must be at least 256");

        // マージの総数
        let num_merges = vocab_size - 256;
        log::info!("Starting BPE training: {} merges to compute", num_merges);

        // マージルールを初期化
        self.merges.clear();

        // Wordのリストとその出現回数から、ペアの出現回数とそのペアが含まれるワードのインデックスを計算
        log::info!("Computing initial pair counts from {} unique sequences", words.len());
        let (mut pair_counts, mut where_to_update) = count_pairs_parallel(&words, &counts);

        // 8分ヒープを初期化
        log::info!("Building heap with {} unique pairs", pair_counts.len());
        let mut heap = OctonaryHeap::with_capacity(pair_counts.len());

        // ペアが含まれるWordのインデックスのリストをループ
        for (pair, pos) in where_to_update.drain() {

            // ペアの総出現回数を取得
            let c = *pair_counts.get(&pair).unwrap_or(&0);

            // ペアの総出現回数が0より大きい場合
            if c > 0 {
                // マージジョブをヒープに追加（出現回数の多い順にマージ可能になる）
                heap.push(MergeJob {
                    pair,
                    count: c as u64,
                    pos,
                });
            }
        }

        // ---- Merge loop ----
        log::info!("Starting merge loop");
        let mut merges_done = 0u32;
        let mut last_log_percent = 0u32;

        // 総マージ数になるまで繰り返す
        while merges_done < num_merges {

            // ヒープから優先度の高いマージジョブを取得
            let Some(mut top) = heap.pop() else { break; };

            // 現在のペアの出現回数を取得
            let current = *pair_counts.get(&top.pair).unwrap_or(&0);

            // マージジョブにある出現回数と一致しない場合、更新しループをやり直す
            if top.count != current as u64 {
                top.count = current as u64;
                if top.count > 0 {
                    heap.push(top);
                }
                continue;
            }

            // ペアの出現回数が0の場合は終了
            if top.count == 0 {
                break;
            }

            // Record merge
            let new_id = 256 + merges_done;
            self.merges.insert(top.pair, new_id);

            // Merge this pair in all words where it occurs
            let mut local_pos_updates: AHashMap<Pair, AHashSet<usize>> = AHashMap::new();

            // ペアが含まれるWordのみをマージ処理
            for &word_idx in &top.pos {

                // Wordに含まれるペアに対しマージを実行
                let changes = words[word_idx].merge_pair(top.pair, new_id);

                // delta（ペアの増減）を使用してpair_countsを更新
                for (pair, delta) in changes {
                    let delta_total = delta * counts[word_idx];
                    if delta_total != 0 {
                        // pair_countsを更新
                        *pair_counts.entry(pair).or_default() += delta_total;

                        if delta > 0 {
                            // 新しいペアが含まれるWordのインデックスを記録
                            local_pos_updates.entry(pair).or_default().insert(word_idx);
                        }
                    }
                }
            }

            // 新しく追加したペアのマージジョブをヒープに追加
            for (pair, pos) in local_pos_updates {
                // ペアの数を取得
                let cnt = *pair_counts.get(&pair).unwrap_or(&0);

                if cnt > 0 {
                    // マージジョブを作成し追加
                    heap.push(MergeJob {
                        pair,
                        count: cnt as u64,
                        pos,
                    });
                }
            }

            // マージ完了数を1追加
            merges_done += 1;

            // 進捗状況を表示
            let current_percent = (merges_done * 100) / num_merges;
            if current_percent > last_log_percent {
                log::info!(
                    "Progress: {}% ({}/{} merges) - Last merge: {:?} -> {} (frequency: {})",
                    current_percent, merges_done, num_merges, top.pair, new_id, top.count
                );
                last_log_percent = current_percent;
            }
        }

        log::info!("Finished training: {} merges completed", merges_done);
    }
}
```

### 5. Tokenizer（Python公開用）

```rust
#[pymethods] // Pythonからtokenizer.関数名のように呼び出し可能にするアトリビュート
impl Tokenizer {

    // コンストラクタ
    // Pythonでrustbpe.Tokenizer()を実行すると呼び出される
    #[new]
    pub fn new() -> Self {
        Self {
            merges: StdHashMap::new(),
            pattern: String::new(),
            compiled_pattern: Regex::new("").expect("Empty regex should be valid"),
        }
    }

    // 訓練の実行メソッド
    #[pyo3(signature = (iterator, vocab_size, buffer_size=8192, pattern=None))]
    #[pyo3(text_signature = "(self, iterator, vocab_size, buffer_size=8192, pattern=None)")]
    pub fn train_from_iterator(
        &mut self,
        py: pyo3::Python<'_>,
        iterator: &pyo3::Bound<'_, pyo3::PyAny>, // Pythonのイテレータ
        vocab_size: u32, // 語彙サイズ
        buffer_size: usize, // テキスト読み込みのバッファサイズ（8192）
        pattern: Option<String>, // 正規表現
    ) -> PyResult<()> {

        // 正規表現を設定
        let pattern_str = pattern.unwrap_or_else(|| GPT4_PATTERN.to_string());
        self.pattern = pattern_str.clone();

        // 正規表現をコンパイル
        self.compiled_pattern = Regex::new(&pattern_str)
            .map_err(|e| pyo3::exceptions::PyValueError::new_err(format!("Invalid regex pattern: {}", e)))?;

        // Pythonのイテレータを読み込む準備
        let py_iter: pyo3::Py<pyo3::PyAny> = unsafe {
            pyo3::Py::from_owned_ptr_or_err(py, pyo3::ffi::PyObject_GetIter(iterator.as_ptr()))?
        };

        // Global chunk counts
        // テキストチャンクと
        let mut counts: AHashMap<CompactString, i32> = AHashMap::new();

        // Rustにテキストを読み込む際のバッファー
        let mut buf: Vec<String> = Vec::with_capacity(buffer_size);

        log::info!("Processing sequences from iterator (buffer_size: {})", buffer_size);
        let mut total_sequences = 0u64;

        // Pythonのイテレータからテキストを読み込む関数
        let refill = |buf: &mut Vec<String>| -> PyResult<bool> {
            // with_gilでPythonのイテレータにアクセス
            pyo3::Python::with_gil(|py| {

                buf.clear();

                let it = py_iter.bind(py);

                loop {
                    if buf.len() >= buffer_size {
                        return Ok(false);
                    }
                    // next(it)
                    let next_obj = unsafe {
                        pyo3::Bound::from_owned_ptr_or_opt(py, pyo3::ffi::PyIter_Next(it.as_ptr()))
                    };
                    match next_obj {
                        Some(obj) => {
                            let s: String = obj.extract()?;
                            buf.push(s);
                        }
                        None => {
                            if pyo3::PyErr::occurred(py) {
                                return Err(pyo3::PyErr::fetch(py));
                            } else {
                                return Ok(true); // exhausted
                            }
                        }
                    }
                }
            })
        };

        // ストリーム処理
        loop {
            // バッファに読み込み
            let exhausted = refill(&mut buf)?;

            if buf.is_empty() && exhausted {
                break;
            }

            total_sequences += buf.len() as u64;

            let pattern = self.compiled_pattern.clone();

            // 読み込んだバッファを全CPUコアで並列に事前トークン化
            let local: AHashMap<CompactString, i32> = py.allow_threads(|| {
                buf.par_iter()
                    .map(|s| {
                        let mut m: AHashMap<CompactString, i32> = AHashMap::new();
                        for mat in pattern.find_iter(s) {
                            let piece = mat.expect("regex match failed").as_str();
                            *m.entry(CompactString::from(piece)).or_default() += 1;
                        }
                        m
                    })
                    .reduce(
                        || AHashMap::new(),
                        |mut a, b| {
                            for (k, v) in b {
                                *a.entry(k).or_default() += v;
                            }
                            a
                        },
                    )
            });

            // Wordの出現回数を加算
            for (k, v) in local {
                *counts.entry(k).or_default() += v;
            }

            if exhausted {
                break;
            }
        }
        log::info!("Processed {} sequences total, {} unique", total_sequences, counts.len());

        // Wordのリストとその出現回数のリストを作成
        let mut words = Vec::with_capacity(counts.len());
        let mut cvec = Vec::with_capacity(counts.len());
        for (chunk, c) in counts.into_iter() {
            words.push(Word::new(chunk.as_bytes().iter().map(|&b| b as u32).collect()));
            cvec.push(c);
        }

        // BPEの訓練を実行
        self.train_core_incremental(words, cvec, vocab_size);
        Ok(())
    }

    // 正規表現のゲッター
    pub fn get_pattern(&self) -> String {
        self.pattern.clone()
    }

    // マージルール（mergeable ranks）のゲッター
    pub fn get_mergeable_ranks(&self) -> Vec<(Vec<u8>, u32)> {

        // 初期化
        let mut mergeable_ranks = Vec::new();

        // 0から255のバイト値をマージルールに追加
        let mut token_bytes: Vec<Vec<u8>> = (0..256_u32).map(|i| vec![i as u8]).collect();
        for (i, bytes) in token_bytes.iter().enumerate() {
            mergeable_ranks.push((bytes.clone(), i as u32));
        }

        // self.mergesをトークンIDの昇順に並び替える
        let mut sorted_merges: Vec<_> = self.merges.iter().collect();
        sorted_merges.sort_by_key(|&(_, &token_id)| token_id);

        // ソートした結果からマージルールに追加
        for (&pair, &merged_id) in sorted_merges {

            // ペアのトークンIDを取得
            let (left, right) = pair;

            // ペアをバイト列に変換し、マージ
            let mut merged_bytes = token_bytes[left as usize].clone();
            merged_bytes.extend(&token_bytes[right as usize]);

            if token_bytes.len() <= merged_id as usize {
                token_bytes.resize(merged_id as usize + 1, Vec::new());
            }

            // 新しいマージIDにマージしたバイト列を追加
            token_bytes[merged_id as usize] = merged_bytes.clone();

            // マージルールに追加
            mergeable_ranks.push((merged_bytes, merged_id));
        }

        mergeable_ranks
    }

    // 文字列をトークンIDに変換するメソッド
    pub fn encode(&self, text: &str) -> Vec<u32> {
        let mut all_ids = Vec::new();

        // 正規表現で事前正規化
        for m in self.compiled_pattern.find_iter(text) {

            // テキストチャンクを取得
            let chunk = m.expect("regex match failed").as_str();

            // テキストチャンクをバイト列に変換
            let mut ids: Vec<u32> = chunk.bytes().map(|b| b as u32).collect();

            // マージを開始
            while ids.len() >= 2 {

                // マージ対象のペアの変数を初期化
                let mut best_pair: Option<(usize, Pair, u32)> = None;

                // すべてのペアで検証
                for i in 0..ids.len() - 1 {
                    let pair: Pair = (ids[i], ids[i + 1]);

                    // トークンIDを取得し、IDが最も小さい場合は更新
                    if let Some(&new_id) = self.merges.get(&pair) {
                        if best_pair.is_none() || new_id < best_pair.unwrap().2 {
                            best_pair = Some((i, pair, new_id));
                        }
                    }
                }

                // マージするペアが見つかった場合、マージを実行
                if let Some((idx, _pair, new_id)) = best_pair {
                    ids[idx] = new_id;
                    ids.remove(idx + 1);
                } else {
                    break;
                }
            }

            all_ids.extend(ids);
        }

        all_ids
    }
}
```

### 6. rustbpe

```rust
#[pymodule] // import rustbpe のように読み込み可能にする関数
fn rustbpe(m: &Bound<'_, PyModule>) -> PyResult<()> {
    pyo3_log::init(); // Pythonのloggerと同期
    m.add_class::<Tokenizer>()?; // #[pyclass]のついたトークナイザークラスを公開
    Ok(())
}
```

### ビルド

[maturin][1]（マチュリン）は、Rustで書いたコードをPythonからimport可能にするビルダー

[1]: https://www.maturin.rs/tutorial.html

In [ ]:
# !maturin develop --release --manifest-path nanochat/rustbpe/Cargo.toml

### 訓練

In [ ]:
import rustbpe

rustbpe_tokenizer = rustbpe.Tokenizer()
rustbpe_tokenizer.train_from_iterator([long_text], vocab_size=276)

In [ ]:
rustbpe_tokenizer.encode("吾輩は猫である。")

In [ ]:
rustbpe_tokenizer.get_pattern()

## ベンチマーク

英語版のWikipediaデータ[enwiki8][1]を使用して検証

- UTF-8エンコードされたXML
- enwiki8.zipは、36MB
- enwiki9.zipは、323MB

[1]: https://mattmahoney.net/dc/textdata.html

In [ ]:
def get_base_dir():
    "キャッシュディレクトリのパスを取得する"
    home_dir = os.path.expanduser("~")
    cache_dir = os.path.join(home_dir, ".cache")
    nanochat_dir = os.path.join(cache_dir, "nanochat")
    os.makedirs(nanochat_dir, exist_ok=True)
    return nanochat_dir

In [ ]:
def enwik8_path():
    "enwik8データセットをダウンロードし、パスを返す"
    base_dir = get_base_dir()
    enwik8_url = "https://mattmahoney.net/dc/enwik8.zip"
    enwik8_local_path = os.path.join(base_dir, "enwik8")
    enwik8_local_path_zip = os.path.join(base_dir, "enwik8.zip")
    if not os.path.exists(enwik8_local_path):
        print(f"Downloading enwik8 to {enwik8_local_path_zip}")
        import requests
        response = requests.get(enwik8_url)
        with open(enwik8_local_path_zip, "wb") as f:
            f.write(response.content)
        with zipfile.ZipFile(enwik8_local_path_zip, "r") as zip_ref:
            zip_ref.extractall(base_dir)
        print(f"Unzipped enwik8 to {enwik8_local_path}")
        os.remove(enwik8_local_path_zip)
        print(f"Removed {enwik8_local_path_zip}")
    else:
        print(f"Using existing enwik8 at {enwik8_local_path}")
    return enwik8_local_path

enwik8_path()

In [ ]:
def enwik8_small(enwik8_path):
    "100KBのenwik8を提供する"
    with open(enwik8_path, "r", encoding="utf-8") as f:
        return f.read(100_000)

In [ ]:
def enwik8_large(enwik8_path):
    "10MBのenwik8を提供する関数"
    with open(enwik8_path, "r", encoding="utf-8") as f:
        return f.read(10**7)

In [ ]:
def time_function(func, *args, **kwargs):
    "関数funcの実行時間を測定する"
    start_time = time.time()
    result = func(*args, **kwargs)
    end_time = time.time()
    elapsed = end_time - start_time
    return result, elapsed

enwik8_smallでの訓練時間を比較

In [ ]:
regex_tokenizer = RegexTokenizer()
time_function(regex_tokenizer.train, enwik8_small(enwik8_path()), vocab_size=2048)

In [ ]:
rustbpe_tokenizer = rustbpe.Tokenizer()
time_function(rustbpe_tokenizer.train_from_iterator, [enwik8_small(enwik8_path())], vocab_size=2048)

enwik8_largeでのエンコード時間を比較

In [ ]:
regex_result = time_function(regex_tokenizer.encode, enwik8_large(enwik8_path()))
len(regex_result[0]), regex_result[1]

In [ ]:
rustbpe_result = time_function(rustbpe_tokenizer.encode, enwik8_large(enwik8_path()))
len(rustbpe_result[0]), rustbpe_result[1]